# 第10章 ロボットの手先を動かして未来を見る (UMI の Forward Dynamics)

第8章では車を動かしました。この章では**ロボットの手先 (グリッパー)** を動かします。
使うのは UMI (Universal Manipulation Interface) という、手持ちのグリッパーにカメラを付けて作業を記録する仕組みのデータです。
グリッパーに付いたカメラの映像なので、手先が動くとカメラごと景色が動きます。

1. ロボットの行動の形を知る (10 個の数: 手先の位置と向き 9 個 + グリッパーの開き 1 個)
2. 16 ステップずつ、つなげて予測する (自己回帰ロールアウト)
3. 行動を書き換えて「もしも」を見比べる (止まる・左右反転・動きを半分)
4. SO-101 など、対応していないロボットで使うには

設定は公式の推奨どおりです: 16 ステップの行動 (20fps)、256p、`num_inference_steps=30`、`guidance_scale=1.0`、`flow_shift=10.0`。
256p で短いので、第8章よりずっと速く終わります。

## 準備

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください (T4 は対象外)。
A100 (40GB 以上) ではモデル全体を GPU に載せ、L4 (24GB) では使う部品だけを GPU に載せる設定 (CPU オフロード) に自動で切り替えます。

**安全フィルタ**: 生成には NVIDIA の安全フィルタ (`cosmos_guardrail`) を使います。使うには Hugging Face で
[nvidia/Cosmos-1.0-Guardrail](https://huggingface.co/nvidia/Cosmos-1.0-Guardrail) の利用申請をして承認を受け、
Colab の左の鍵のマーク (シークレット) に `HF_TOKEN` を登録しておいてください。承認を待たずに試すときは、下の `GUARDRAIL` の印を外します。

In [ ]:
# @title GPU を確かめる
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(torch.cuda.get_device_name(0), f"{GPU_GB:.0f} GB")
    assert torch.cuda.get_device_capability()[0] >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"
else:
    GPU_GB = 0
    print(f"CUDA の GPU が見つかりません ({DEVICE} で動かします。とても遅くなります)")

In [ ]:
# @title インストール (数分)
# Cosmos3OmniPipeline は Diffusers の main ブランチに入っている
!pip install -q -U "transformers>=5.18" accelerate av imageio imageio-ffmpeg huggingface_hub
!pip install -q "diffusers @ git+https://github.com/huggingface/diffusers.git" cosmos_guardrail

In [ ]:
# @title 安全フィルタの設定とサンプル素材を取ってくる関数
import os
import urllib.request
from pathlib import Path

GUARDRAIL = True  # @param {type:"boolean"}
try:  # Colab のシークレットに HF_TOKEN があれば使う
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

ASSETS = Path(os.environ.get("COSMOS_ASSETS", "assets"))
ASSETS.mkdir(parents=True, exist_ok=True)
FW = "https://media.githubusercontent.com/media/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
HF = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets"


def fetch(url):
    """URL の素材を assets/ に落として、そのパスを返す (2 回目からは落とさない)"""
    path = ASSETS / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return str(path)

In [ ]:
# @title 生成モデル (Generator) を読み込む関数
import gc
import json

from diffusers import Cosmos3OmniPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageOps


def load_image(path, max_side=10_000):
    """画像を RGB で読む (max_side より大きければ縮める)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def load_pipe(flow_shift):
    """Cosmos3OmniPipeline を読み込む (2 回目からは読み込み済みのものを使う)。
    flow_shift は公式の推奨値: 画像→動画は 12.0、行動 (action) を使う生成は 10.0"""
    global pipe
    if "pipe" not in globals():
        pipe = Cosmos3OmniPipeline.from_pretrained(
            "nvidia/Cosmos3-Edge", torch_dtype=torch.bfloat16, enable_safety_checker=GUARDRAIL)
        if DEVICE == "cuda" and GPU_GB < 40:
            pipe.enable_model_cpu_offload()  # L4 など: 使う部品だけ GPU に載せる
        else:
            pipe.to(DEVICE)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=flow_shift, use_karras_sigmas=False)
    return pipe


def free_gpu():
    """読み込んだモデルを GPU から降ろす"""
    for name in ("pipe", "model", "processor"):
        globals().pop(name, None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def show_video(path, width=640):
    display(Video(path, embed=True, width=width))

In [ ]:
# @title 行動 (action) を作る・描く関数
import numpy as np
import matplotlib.pyplot as plt
from diffusers import CosmosActionCondition

RAW = "https://raw.githubusercontent.com/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
AV_PROMPT = "You are an autonomous vehicle planning system."
# AV の行動は「1 コマ前のカメラから見た、次のコマのカメラの位置と向き」を 9 個の数で表す。
#   [x, y, z, r1..r6]  x: 右, y: 下, z: 前 (OpenCV のカメラ座標)。r1..r6: 回転行列の 1 列目と 2 列目 (rot6d)
# 移動量はメートルを 1.35 倍した値 (公式の可視化コードと同じ換算)
AV_SCALE = 1.35


def rot_y(theta):
    """カメラの下向きの軸 (y) まわりの回転。theta が負だと左を向く"""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def from_rot6d(r):
    a = np.asarray(r[:3], float)
    b = np.asarray(r[3:6], float)
    c0 = a / np.linalg.norm(a)
    c1 = b - c0 * (c0 @ b)
    c1 /= np.linalg.norm(c1)
    return np.stack([c0, c1, np.cross(c0, c1)], axis=1)


def make_av_actions(speed_kmh=30.0, left_deg_per_s=0.0, accel=0.0, seconds=6.0, fps=10):
    """速さ (km/h)・左へ曲がる速さ (度/秒、右は負)・加速度 (m/s^2、減速は負) から AV の行動 (T, 9) を作る。
    left_deg_per_s には「時刻 t (秒) → 度/秒」の関数も渡せる (車線変更などに)"""
    rows, dt = [], 1 / fps
    for i in range(int(round(seconds * fps))):
        t = i * dt
        v = max(speed_kmh / 3.6 + accel * t, 0.0)
        d = v * dt
        yaw = left_deg_per_s(t) if callable(left_deg_per_s) else left_deg_per_s
        th = -np.radians(yaw) * dt if d > 0 else 0.0
        move = np.array([np.sin(th / 2) * d, 0.0, np.cos(th / 2) * d]) * AV_SCALE
        R = rot_y(th)
        rows.append(np.concatenate([move, R[:, 0], R[:, 1]]))
    return np.array(rows, np.float32)


def to_path(actions, scale=AV_SCALE):
    """行動 (T, 9) をつなげて、上から見た道筋 (T+1, 3) にする。x: 右, z: 前 (メートル)"""
    R, p, pts = np.eye(3), np.zeros(3), [np.zeros(3)]
    for a in np.asarray(actions, float):
        p = p + R @ (a[:3] / scale)
        R = R @ from_rot6d(a[3:9])
        pts.append(p.copy())
    return np.array(pts)


def plot_paths(paths, title="top-down view", fps=10):
    """{名前: 行動 (T, 9)} を上から見た図に描く。点は 1 秒ごと"""
    fig, ax = plt.subplots(figsize=(5, 5))
    for i, (name, actions) in enumerate(paths.items()):
        p = to_path(actions)
        ax.plot(p[:, 0], p[:, 2], lw=2, label=name)
        ax.scatter(p[::fps, 0], p[::fps, 2], s=12)
    ax.scatter([0], [0], c="k", marker="^", s=80, zorder=5)
    ax.set_xlabel("x: right (m)")
    ax.set_ylabel("z: forward (m)")
    ax.set_aspect("equal", adjustable="datalim")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    ax.set_title(title)
    plt.show()


def tile_videos(videos, labels, out, fps, cols=2):
    """同じ長さの動画 (PIL 画像のリスト) を並べて 1 本にする"""
    from PIL import ImageDraw, ImageFont
    n = min(len(v) for v in videos)
    w, h = videos[0][0].size
    rows = (len(videos) + cols - 1) // cols
    font = ImageFont.load_default(size=max(16, w // 25))
    frames = []
    for t in range(n):
        canvas = Image.new("RGB", (w * cols, h * rows))
        for k, (v, label) in enumerate(zip(videos, labels)):
            tile = v[t].convert("RGB").resize((w, h))
            ImageDraw.Draw(tile).text((10, 8), label, fill="white", font=font, stroke_width=3, stroke_fill="black")
            canvas.paste(tile, ((k % cols) * w, (k // cols) * h))
        frames.append(canvas)
    export_to_video(frames, out, fps=fps, macro_block_size=1)
    return out

## 1. ロボットの行動の形を知る

モデルカードのお手本 (マウスを並べる作業) の行動を読み込みます。16 ステップのかたまり (チャンク) が 2 つです。

| 位置 | 意味 |
| --- | --- |
| 0〜2 | 手先 (カメラ) の位置。**そのチャンクの最初のコマから見た**位置 |
| 3〜8 | 手先の向き (rot6d)。同じく最初のコマから見た向き |
| 9 | グリッパーの開き (正規化した値) |

車 (第8章) は「1 コマ前から見た変化」でしたが、UMI は「チャンクの最初から見た位置」で表します。行動の表し方はロボットの種類ごとに決まっています。

In [ ]:
spec = json.loads(Path(fetch(f"{HF}/example_action_fd_umi_action_chunks.json")).read_text())
chunks = [np.array(c, np.float32) for c in spec["action_chunks"]]
first = load_image(fetch(f"{HF}/example_action_fd_umi_first_frame.png"), max_side=10_000)
print("prompt:", spec["prompt"], "/ fps:", spec["fps"], "/ チャンク:", [c.shape for c in chunks])
display(first)


def umi_path(chunks):
    """チャンクごとの行動をつなげて、手先の位置の並びにする (単位は正規化した値)"""
    A, pts = np.eye(4), [np.zeros(3)]
    for c in chunks:
        for a in c:
            T = np.eye(4)
            T[:3, :3], T[:3, 3] = from_rot6d(a[3:9]), a[:3]
            pts.append((A @ T)[:3, 3])
        A = A @ T  # 次のチャンクは、このチャンクの最後の姿勢から
    return np.array(pts)


def plot_umi(named_chunks, title):
    fig = plt.figure(figsize=(11, 4))
    ax = fig.add_subplot(1, 2, 1, projection="3d")
    ax2 = fig.add_subplot(1, 2, 2)
    for name, cs in named_chunks.items():
        p = umi_path(cs)
        ax.plot(p[:, 0], p[:, 2], -p[:, 1], label=name)
        ax2.plot(np.concatenate([c[:, 9] for c in cs]), label=name)
    ax.set_xlabel("x right")
    ax.set_ylabel("z forward")
    ax.set_zlabel("up")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax2.set_title("gripper (normalized)")
    ax2.set_xlabel("step")
    plt.show()


plot_umi({"official": chunks}, "end-effector path")

## 2. 16 ステップずつ、つなげて予測する

1 回の生成で作れるのは 16 ステップ (17 コマ) です。もっと先まで予測するときは、**前のチャンクの最後のコマを、次のチャンクの最初のコマにして**つなげます。
これを自己回帰ロールアウトと呼びます。つなぐたびに少しずつ誤差がたまるのが弱点です (公式の注意にある action-state drift)。

In [ ]:
def rollout(image, chunks, prompt=spec["prompt"], seed=0):
    """チャンクのリストを順に生成してつなげ、動画 (PIL のリスト) を返す。各チャンクの最初のコマ (前の続き) は捨てる"""
    pipe = load_pipe(flow_shift=10.0)
    frames, current = [image], image
    for k, c in enumerate(chunks):
        result = pipe(
            prompt=prompt,
            action=CosmosActionCondition(
                mode="forward_dynamics",
                chunk_size=len(c),
                domain_name="umi",
                resolution_tier=256,
                raw_actions=torch.as_tensor(c, dtype=torch.float32),
                image=current,
                view_point="ego_view",
            ),
            fps=spec["fps"],
            num_inference_steps=30,
            guidance_scale=1.0,
            use_system_prompt=False,
            generator=torch.Generator(device=DEVICE).manual_seed(seed + k),
        )
        frames += result.video[1:]
        current = result.video[-1]
    return frames


official_video = rollout(first, chunks)
export_to_video(official_video, "ch10_official.mp4", fps=spec["fps"], macro_block_size=1)
show_video("ch10_official.mp4", 384)

## 3. 行動を書き換えて「もしも」を見比べる

同じ最初のコマから、行動だけを書き換えて予測させます。

- **stay**: 手先を動かさない (位置 0、向きは変えない)。景色が止まっていれば正しい
- **mirror**: 左右反転した動き。景色が逆向きに流れれば正しい
- **half**: 動きを半分にする。景色の動きも半分になれば正しい

In [ ]:
S = np.diag([-1.0, 1.0, 1.0])
IDENTITY = np.array([0, 0, 0, 1, 0, 0, 0, 1, 0], np.float32)


def edit(chunks, how):
    out = []
    for c in chunks:
        c2 = c.copy()
        for i, a in enumerate(c):
            R = from_rot6d(a[3:9])
            if how == "stay":
                c2[i, :9] = IDENTITY
            elif how == "mirror":
                R = S @ R @ S
                c2[i, :9] = np.concatenate([S @ a[:3], R[:, 0], R[:, 1]])
            elif how == "half":
                c2[i, :3] = a[:3] * 0.5
        out.append(c2)
    return out


variants = {"official": chunks, "stay": edit(chunks, "stay"), "mirror": edit(chunks, "mirror"), "half": edit(chunks, "half")}
plot_umi(variants, "edited actions")
videos = [official_video] + [rollout(first, cs) for name, cs in variants.items() if name != "official"]
tile_videos(videos, list(variants), "ch10_what_if.mp4", fps=spec["fps"])
show_video("ch10_what_if.mp4", 640)

## 4. SO-101 など、対応していないロボットで使うには

Edge がそのまま扱える行動の形は、次のロボットに限られます (モデルカードより)。

| 種類 | 次元 |
| --- | --- |
| カメラの動き / 自動運転 | 9 |
| 一人称の手の動き | 57 |
| Franka Panda + RobotiQ (片腕 / 双腕) | 10 / 20 |
| UR / Google robot / WidowX 250 / UMI | 10 |
| Agibot | 29 |

SO-101 のような手元のロボットで「この関節の動かし方をしたら、カメラにどう映るか」を予測させるには、
LeRobot で撮った自分のデータで**追加学習 (ポストトレーニング)** をするのが公式の道筋です
(Cosmos Framework の forward dynamics のポストトレーニングのレシピ。DROID での例があります)。
その前段として、手元の GPU 1 枚で小さな世界モデルを 0 から作る例は、同じフォルダの `world_model_so101.ipynb` を見てください。

## うまくいかないとき

- **つなぐほど崩れる**: 自己回帰のつなぎ目で誤差がたまります。チャンクの数を減らすか、seed を変えて作り直します
- **stay でも少し動く**: 行動に関係なく「もっともらしい動き」を描いてしまうことがあります。行動がどれだけ効いているかは、こうした書き換えの比較で確かめるのが確実です
- **自分の UMI データで試したい**: 行動は正規化した値です。正規化の統計は Cosmos Framework の `normalizer_stats` にあります